In [1]:
import numpy as np
from scipy.io import wavfile
import IPython.display as ipd
from scipy import signal as sig
from scipy.linalg import solve_toeplitz
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
%config InlineBackend.figure_format = 'svg'

import Funciones as Funciones
from NNMF_GIF import NNMF_GIF
from MetricasRendimiento import MetricasRendimiento
import os
import matplotlib 
matplotlib.use("Agg")

# Carga de la señal

In [2]:
with open("Señales Repositorio II Resampleadas/Listas señales/Completa todas.txt", "r") as f:
    archivos = [line.strip() for line in f]
#archivos = archivos[0]

fs_r = 8000
cmap = Funciones.parula_map
color_flujo = 'blue'
color_dflujo = 'red'
color_area = 'orange'
color_darea = 'g'
color_energia = 'magenta'
color_speech = 'black'
legend_loc = 'upper right'



In [3]:
error_1 = []
error_2 = []
etiqueta = []

# Definición de parámetros

In [4]:
norma ='fro'
pre_iteraciones = 0
iteraciones     = 250
p_filtro        = 12
p = 0.55
pre_enfasis     = True
duracion        = 5.5
guardar_proceso = True

In [5]:
archivo = archivos[3]
señales       = np.loadtxt("Señales Repositorio II Resampleadas/"+archivo+".txt", delimiter="\t")
tiempos_      = señales[:, 0]
señal_speech_ = señales[:, 1]
señal_flujo_  = señales[:, 2]
señal_dflujo_ = señales[:, 3]
señal_area_   = señales[:, 4]
señal_darea_  = señales[:, 5]
to = 750
tf = 800
no = int(np.round(to *fs_r/1000)) 
nf = int(np.round(tf * fs_r/1000))
tiempos       = tiempos_[no:nf]

señal_speech  = señal_speech_[no:nf]
señal_flujo   = señal_flujo_[no:nf]
señal_dflujo   = señal_dflujo_[no:nf]
señal_area    = señal_area_[no:nf]
señal_darea   = señal_darea_[no:nf]

SP1 = NNMF_GIF(señal_speech,tiempos,fs_r,
        norma=norma,
        pre_iteraciones=pre_iteraciones,
        iteraciones=iteraciones,
        p_filtro=p_filtro,
        p=p,
        pre_enfasis=pre_enfasis,
        duracion=duracion,
        guardar_proceso=guardar_proceso)

señal_flujo  = señal_flujo[SP1.orden_filtro_tracto-1:len(señal_flujo)-SP1.orden_filtro_tracto-1]
señal_dflujo = señal_dflujo[SP1.orden_filtro_tracto-1:len(señal_dflujo)-SP1.orden_filtro_tracto-1]


# Gráficas

In [6]:

# Crear el directorio de salida de forma segura
output_dir = f"Prueba Repositorio II/Animacion/Fro_/{archivo}"
os.makedirs(output_dir, exist_ok=True)

# Calcular valores constantes una sola vez antes de que comience el bucle
fftseñal, fseñal = Funciones.FFT(señal_speech, fs_r)
iteraciones_range = np.arange(iteraciones)
theta_circle = np.linspace(0, 2 * np.pi, 200)
unit_circle_x = np.cos(theta_circle)
unit_circle_y = np.sin(theta_circle)

# Constantes de estilo para los gráficos
color1 = 'red'
color2 = 'blue'
color_error = 'green'
loc = 'lower right'

# --- 2. FUNCIÓN AUXILIAR PARA EVITAR CÓDIGO REPETIDO ---

def plot_error_evolution(ax, x_data, y_data, current_iter, color, label_prefix, precision=7):
    """
    Dibuja la evolución de una métrica de error hasta la iteración actual.
    Reduce la duplicación de código en la sección de gráficos de error.
    """
    label = f"{label_prefix}({current_iter})={y_data[current_iter]:.{precision}f}"
    if current_iter == 0:
        ax.scatter(x_data[current_iter], y_data[current_iter], s=1, color=color, label=label)
    else:
        ax.plot(x_data[:current_iter + 1], y_data[:current_iter + 1], color=color, label=label)
    
    ax.set_xlabel("Iteración n°")
    ax.set_yticks([])
    ax.set_xlim(-1, current_iter + 1 if current_iter > 0 else 1)
    ax.legend(loc='upper right', fontsize=8)

# --- 3. BUCLE PRINCIPAL DE PROCESAMIENTO Y GRÁFICOS ---

for i in range(iteraciones):
    # --- Cálculos específicos de la iteración ---
    fw1, hw1 = sig.freqz([1], SP1.proceso_a_W1[i], fs=fs_r)
    fw2, hw2 = sig.freqz([1], SP1.proceso_a_W2[i], fs=fs_r)

    tiempo_flujo1, _, flujo1 = Funciones.Sincronizar2(SP1.tiempos_gve, señal_flujo, SP1.proceso_flujo1[i])
    tiempo_flujo2, _, flujo2 = Funciones.Sincronizar2(SP1.tiempos_gve, señal_flujo, SP1.proceso_flujo2[i])

    tiempo_dflujo1, señal_dflujo_1, dflujo1 = Funciones.Sincronizar2(SP1.tiempos_gve, señal_dflujo, SP1.proceso_dflujo1[i])
    tiempo_dflujo2, señal_dflujo_2, dflujo2 = Funciones.Sincronizar2(SP1.tiempos_gve, señal_dflujo, SP1.proceso_dflujo2[i])

    MR1 = MetricasRendimiento(señal_dflujo_1, Funciones.Escalar(dflujo1, señal_dflujo_1), fs_r, tiempo_dflujo1)
    MR2 = MetricasRendimiento(señal_dflujo_2, Funciones.Escalar(dflujo2, señal_dflujo_2), fs_r, tiempo_dflujo2)

    # --- Creación de la figura y los ejes ---
    fig = plt.figure(figsize=(12, 8))
    widths = [0.6, 0.6, 0.6, 0.6, 0.6, 0.6]
    heights = [1, 1, 1, 0.55]
    gs = fig.add_gridspec(4, 6, width_ratios=widths, height_ratios=heights, wspace=0.15, hspace=0.55)
    
    ax0 = fig.add_subplot(gs[0, 0:3])   # Espectrograma original
    ax1 = fig.add_subplot(gs[1, 0:3])   # Espectrograma aproximado
    ax2 = fig.add_subplot(gs[2, 0:3])   # Plot de H
    ax3 = fig.add_subplot(gs[0, 3:5])   # Plot de W
    ax4 = fig.add_subplot(gs[0, 5:6])   # Polos y ceros
    ax5 = fig.add_subplot(gs[1, 3:6])   # Flujos
    ax6 = fig.add_subplot(gs[2, 3:6])   # Derivadas
    ax7 = fig.add_subplot(gs[3, 0:2])   # Error X
    ax8 = fig.add_subplot(gs[3, 2:4])   # Error W
    ax9 = fig.add_subplot(gs[3, 4:6])   # Error H

    # =============================== Espectrograma original ===============================
    ax0.imshow(SP1.espectrograma, cmap=cmap, origin='lower', aspect='auto', extent=[SP1.to_espec, SP1.tf_espec, SP1.fo_espec, SP1.ff_espec])
    ax0.set_ylabel(r"$f$ [Hz]")
    ax0_ = ax0.twinx()
    ax0_.plot(SP1.tiempos_espec, señal_darea[SP1.long_ventana_espect - 1:], color='white', label=r"$\dfrac{dA(t)}{dt}$", linewidth=2)
    ax0_.legend(loc='upper right')
    ax0_.set_xlim(SP1.to_espec, SP1.tf_espec)
    ax0_.set_xticks(np.arange(SP1.to_espec, SP1.tf_espec, SP1.duracion_ventana))
    ax0_.set_yticks([])
    ax0.set_xlabel('Tiempo [ms]')

    # =============================== Espectrograma aproximado ===============================
    ax1.imshow(SP1.proceso_espectrograma_aprox[i], cmap=cmap, origin='lower', aspect='auto', extent=[SP1.to_espec, SP1.tf_espec, SP1.fo_espec, SP1.ff_espec])
    ax1.set_ylabel(r"$f$ [Hz]")
    ax1_ = ax1.twinx()
    ax1_.plot(tiempos[SP1.long_ventana_espect:], señal_darea[SP1.long_ventana_espect:], color='white', label=r"$\dfrac{dA(t)}{dt}$", linewidth=2)
    ax1_.legend(loc='upper right')
    ax1_.set_xlim(SP1.to_espec, SP1.tf_espec)
    ax1_.set_xticks(np.arange(SP1.to_espec, SP1.tf_espec, SP1.duracion_ventana))
    ax1_.set_yticks([])
    ax1.set_xlabel('Tiempo [ms]')

    # =============================== Funciones de activación ===============================
    ax2.plot(SP1.tiempos_espec, SP1.proceso_H1[i], color=color1, label=r'$H_1$')
    ax2.plot(SP1.tiempos_espec, SP1.proceso_H2[i], color=color2, label=r'$H_2$')
    ax2.set_xlim(SP1.to_espec, SP1.tf_espec)
    ax2.grid(True, linestyle='dashed', alpha=0.3, which='both')
    ax2.legend(loc=loc)
    ax2.set_xlabel('Tiempo [ms]')

    # =============================== Respuesta en frecuencia de W1 y W2 ===============================
    ax3.plot(fw1, 10 * np.log10(np.abs(hw1)), label=r'$W_1$', color=color1)
    ax3.plot(fw2, 10 * np.log10(np.abs(hw2)), label=r'$W_2$', color=color2)
    ax3.plot(fseñal, 10 * np.log10(np.abs(fftseñal)), label='speech', color=color_speech, zorder=-1)
    ax3.set_xlim(0, fs_r / 2)
    ax3.grid(True, linestyle='dashed', alpha=0.3, which='both')
    ax3.set_xlabel('Frecuencia [Hz]')
    ax3.legend(loc='upper right')

    # =============================== Polos y ceros ===============================
    ax4.plot(unit_circle_x, unit_circle_y, 'k--', alpha=1) # Usar datos pre-calculados
    ax4.scatter(np.real(SP1.proceso_ceros_W1[i]), np.imag(SP1.proceso_ceros_W1[i]), s=80, facecolors='none', edgecolors=color1, label='Ceros W1')
    ax4.scatter(np.real(SP1.proceso_polos_W1[i]), np.imag(SP1.proceso_polos_W1[i]), s=80, marker='x', color=color1, label='Polos W1')
    ax4.scatter(np.real(SP1.proceso_ceros_W2[i]), np.imag(SP1.proceso_ceros_W2[i]), s=80, facecolors='none', edgecolors=color2, label='Ceros W2')
    ax4.scatter(np.real(SP1.proceso_polos_W2[i]), np.imag(SP1.proceso_polos_W2[i]), s=80, marker='x', color=color2, label='Polos W2')
    ax4.set_ylabel(r"$Im\{z\}$")
    ax4.set_xlabel(r"$Re\{z\}$")
    ax4.grid(True, linestyle='dashed', alpha=0.3)
    ax4.axis('equal') # Asegura que el círculo unitario se vea como un círculo

    # =============================== Flujos glóticos ===============================
    ax5.plot(SP1.tiempos_gve, señal_flujo, color=color_speech, label=r'$u_g(t)$')
    ax5.plot(tiempo_flujo1, flujo1, color=color1, label=r'$\hat{u}_{g1}(t)$')
    ax5.plot(tiempo_flujo2, flujo2, color=color2, label=r'$\hat{u}_{g2}(t)$')
    ax5.set_xlim(to, tf)
    ax5.grid(True, linestyle='dashed', alpha=0.3, which='both')
    ax5.legend(loc=loc)
    ax5.set_xlabel("Tiempo [ms]")

    # =============================== Derivadas de flujos glóticos ===============================
    ax6.plot(SP1.tiempos_gve, señal_dflujo, color=color_speech, label=r'$v_g(t)$')
    ax6.plot(tiempo_dflujo1, dflujo1, color=color1, label=f'$\\hat{{v}}_{{g1}}(t)$ ~ $E_{{vg}}$={MR1.Error:.5f}')
    ax6.plot(tiempo_dflujo2, dflujo2, color=color2, label=f'$\\hat{{v}}_{{g2}}(t)$ ~ $E_{{vg}}$={MR2.Error:.5f}')
    ax6.set_xlim(to, tf)
    ax6.grid(True, linestyle='dashed', alpha=0.3, which='both')
    ax6.legend(loc=loc)
    ax6.set_xlabel("Tiempo [ms]")

    # =============================== Gráficos de Error (usando la función auxiliar) ===============================
    # Error X
    plot_error_evolution(ax7, iteraciones_range, SP1.error_NNMF, i, 'orange', "$ΔX", precision=5)

    # Error W (dos curvas en el mismo eje)
    plot_error_evolution(ax8, iteraciones_range, SP1.dif_error_W1, i, color1, "$ΔW_1$", precision=7)
    plot_error_evolution(ax8, iteraciones_range, SP1.dif_error_W2, i, color2, "$ΔW_2$", precision=7)
    # Volver a llamar a legend() para que muestre ambas etiquetas actualizadas
    ax8.legend(loc='upper right', fontsize=8)

    # Error H (dos curvas en el mismo eje)
    plot_error_evolution(ax9, iteraciones_range, SP1.dif_error_H1, i, color1, "$ΔH_1$", precision=7)
    plot_error_evolution(ax9, iteraciones_range, SP1.dif_error_H2, i, color2, "$ΔH_2$", precision=7)
    ax9.legend(loc='upper right', fontsize=8)

    # --- Título y guardado de la figura ---
    fig.suptitle(
        f"{archivo} | norma: {norma} | pre-iteraciones: {pre_iteraciones} | iteración n°: {i}",
        fontsize=12
    )
    
    # OPTIMIZACIÓN: Guardar como PNG es generalmente más rápido para fotogramas de animación.
    # Se usa os.path.join para construir la ruta de forma segura.
    output_path = os.path.join(output_dir, f"{i}.png")
    fig.savefig(output_path, bbox_inches='tight', dpi=150) # dpi es opcional, ajusta la resolución

    # Liberar la memoria de la figura actual para evitar consumo excesivo
    plt.close(fig)

In [7]:
len(SP1.dif_error_W2)

250